# Explore the clean ENTSO-E data (Germany)

Built on `data/clean/de_hourly.parquet` and `data/clean/day_flags.parquet` (run `src/clean_entsoe.py` and `src/anomalies.py` first).

**Everything here stops at 2023-12-31.** The loaders in `src/data.py` cut off the 2024+ holdout, which stays unseen until the config is frozen.

In [ ]:
import sys
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

sys.path.insert(0, str(Path("..") / "src"))
from data import load_dev_day_flags, load_dev_hourly

CLEAN_DIR = Path("..") / "data" / "clean"
HOLDOUT_START_YEAR = 2024
NUM_CONTEXT_DAYS = 7
NUM_EXAMPLE_FLAGS = 4

plt.rcParams["figure.figsize"] = (12, 4)
pl.Config.set_tbl_rows(30)

In [ ]:
def add_timestamp(hourly):
    """Add a naive local timestamp (date + hour) for plotting."""
    timestamp = pl.col("date").cast(pl.Datetime) + pl.duration(hours=pl.col("hour"))
    return hourly.with_columns(timestamp.alias("timestamp"))


hourly = add_timestamp(load_dev_hourly())
flags = load_dev_day_flags()
hourly.head()

## Data quality

How each hourly value was obtained (`observed`, `interpolated`, `previous_week`, `missing`), and how many values broke an impossible-value rule. The only gaps are early 2015: prices start on 2015-01-05, and one day of offshore wind has no previous week to copy from.

In [ ]:
fill_report = pl.read_csv(CLEAN_DIR / "fill_report.csv").filter(pl.col("year") < HOLDOUT_START_YEAR)
fill_report

In [ ]:
# Explicit types: with no flags the file is header-only and every column would be read as text.
flag_schema = {"year": pl.Int32, "column": pl.String, "rule": pl.String, "len": pl.UInt32}
flag_counts = pl.read_csv(CLEAN_DIR / "flag_counts.csv", schema=flag_schema)
flag_counts = flag_counts.filter(pl.col("year") < HOLDOUT_START_YEAR)
print(f"impossible-value flags 2015-2023: {flag_counts['len'].sum()}")

## Prices over time

In [ ]:
daily_prices = hourly.group_by("date").agg(pl.col("price").mean()).sort("date")

fig, ax = plt.subplots()
ax.plot(daily_prices["date"], daily_prices["price"], linewidth=0.7)
ax.set_ylabel("EUR/MWh")
ax.set_title("Daily mean day-ahead price, Germany")
plt.show()

In [ ]:
def yearly_price_stats(hourly):
    """Return per-year price statistics, including the number of negative-price hours."""
    return (
        hourly.group_by(pl.col("date").dt.year().alias("year"))
        .agg(
            pl.col("price").mean().alias("mean"),
            pl.col("price").std().alias("std"),
            pl.col("price").quantile(0.01).alias("p01"),
            pl.col("price").median().alias("median"),
            pl.col("price").quantile(0.99).alias("p99"),
            pl.col("price").max().alias("max"),
            (pl.col("price") < 0).sum().alias("negative_hours"),
        )
        .sort("year")
    )


yearly_price_stats(hourly).with_columns(pl.selectors.float().round(1))

The 2021-22 crisis makes errors roughly 10x larger than in 2019. This is why the combiner losses are divided by the seasonal naive's trailing MAE. The average change from the same hour yesterday shows the scale shift:

In [ ]:
# |price(h) - price(h, day before)| is the seasonal naive's error on Tuesday-Friday.
day_over_day = hourly.sort("date", "hour").with_columns(
    (pl.col("price") - pl.col("price").shift(1).over("hour")).abs().alias("change")
)
monthly_change = day_over_day.group_by_dynamic("timestamp", every="1mo").agg(pl.col("change").mean())

fig, ax = plt.subplots()
ax.plot(monthly_change["timestamp"], monthly_change["change"], marker=".")
ax.set_yscale("log")
ax.set_ylabel("EUR/MWh (log scale)")
ax.set_title("Monthly mean |price(h) - price(h, day before)|")
plt.show()

## Daily shape

In [ ]:
def mean_profile_by_day_type(hourly):
    """Return mean price per hour of day, split into weekday, Saturday and Sunday."""
    weekday_number = pl.col("date").dt.weekday()
    day_type = (
        pl.when(weekday_number == 6).then(pl.lit("saturday"))
        .when(weekday_number == 7).then(pl.lit("sunday"))
        .otherwise(pl.lit("weekday"))
    )
    profile = hourly.group_by("hour", day_type.alias("day_type")).agg(pl.col("price").mean())
    return profile.pivot(on="day_type", index="hour", values="price").sort("hour")


periods = {"2017-2019": (2017, 2019), "2020": (2020, 2020), "2022": (2022, 2022)}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (label, (first_year, last_year)) in zip(axes, periods.items()):
    in_period = pl.col("date").dt.year().is_between(first_year, last_year)
    profile = mean_profile_by_day_type(hourly.filter(in_period))
    for day_type in ["weekday", "saturday", "sunday"]:
        ax.plot(profile["hour"], profile[day_type], label=day_type)
    ax.set_title(label)
    ax.set_xlabel("hour of day")
axes[0].set_ylabel("EUR/MWh")
axes[0].legend()
plt.show()

## Prices against fundamentals

Residual load = load forecast - wind - solar: the demand left for thermal plants, which set the price.

In [ ]:
with_residual = hourly.with_columns(
    (pl.col("load_forecast_winsorized") - pl.col("renewables_total")).alias("residual_load")
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, year in zip(axes, [2019, 2021, 2023]):
    year_data = with_residual.filter(pl.col("date").dt.year() == year)
    ax.scatter(year_data["residual_load"] / 1000, year_data["price"], s=1, alpha=0.2)
    ax.set_title(str(year))
    ax.set_xlabel("residual load forecast (GW)")
axes[0].set_ylabel("price (EUR/MWh)")
plt.show()

In [ ]:
with_residual.group_by(pl.col("date").dt.year().alias("year")).agg(
    pl.corr("price", "residual_load").round(2).alias("corr(price, residual_load)")
).sort("year")

## Load winsorizing

Each hour is clipped to the 0.5% and 99.5% quantiles of the same hour over the previous 364 days. It clips 2-4% of hours (more than the nominal 1%, because load drifts between years); kept as is by choice.

In [ ]:
clipped = hourly.with_columns((pl.col("load_forecast") != pl.col("load_forecast_winsorized")).alias("clipped"))
clipped.group_by(pl.col("date").dt.year().alias("year")).agg(
    (pl.col("clipped").mean() * 100).round(1).alias("percent_clipped")
).sort("year")

In [ ]:
winter_week = hourly.filter(pl.col("date").is_between(pl.date(2022, 12, 18), pl.date(2023, 1, 8)))

fig, ax = plt.subplots()
ax.plot(winter_week["timestamp"], winter_week["load_forecast"] / 1000, label="raw", linewidth=1)
ax.plot(winter_week["timestamp"], winter_week["load_forecast_winsorized"] / 1000, label="winsorized", linewidth=1)
ax.set_ylabel("GW")
ax.set_title("Load forecast over Christmas 2022: holiday lows get clipped")
ax.legend()
plt.show()

## Outlier days

A day is an outlier if either detector fires:
- **z-score**: mean |robust z| over all hours of load, wind and solar, above the 99th percentile of the previous 364 days. The MAD is floored at 2% of the series' mean level and each |z| is capped at 10, so near-zero solar hours at dawn and dusk can't dominate.
- **isolation forest**: daily mean/min/max of load, wind, solar and residual load, contamination 0.5%, refit yearly. It also flags record wind/solar days from capacity growth, kept on purpose for diversity.

Outlier days are dropped as training targets but stay in evaluation.

In [ ]:
flags.group_by(pl.col("date").dt.year().alias("year")).agg(
    pl.col("z_flag").sum(),
    pl.col("forest_flag").sum(),
    (pl.col("z_flag") & pl.col("forest_flag")).sum().alias("both"),
    pl.col("is_outlier").sum(),
).sort("year")

In [ ]:
flagged_by_month = flags.filter(pl.col("is_outlier")).group_by(
    pl.col("date").dt.month().alias("month")
).agg(pl.col("z_flag").sum(), pl.col("forest_flag").sum()).sort("month")

fig, ax = plt.subplots()
ax.bar(flagged_by_month["month"] - 0.2, flagged_by_month["z_flag"], width=0.4, label="z-score")
ax.bar(flagged_by_month["month"] + 0.2, flagged_by_month["forest_flag"], width=0.4, label="isolation forest")
ax.set_xticks(range(1, 13))
ax.set_xlabel("month")
ax.set_ylabel("flagged days, 2015-2023")
ax.legend()
plt.show()

In [ ]:
daily_fundamentals = hourly.group_by("date").agg(
    (pl.col("load_forecast").mean() / 1000).alias("load"),
    ((pl.col("wind_onshore") + pl.col("wind_offshore")).mean() / 1000).alias("wind"),
    (pl.col("solar").mean() / 1000).alias("solar"),
).sort("date").join(flags.select("date", "z_flag", "forest_flag"), on="date")

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
for ax, column in zip(axes, ["load", "wind", "solar"]):
    ax.plot(daily_fundamentals["date"], daily_fundamentals[column], linewidth=0.5, color="grey")
    z_days = daily_fundamentals.filter(pl.col("z_flag"))
    forest_days = daily_fundamentals.filter(pl.col("forest_flag"))
    ax.scatter(z_days["date"], z_days[column], s=12, label="z-score", zorder=3)
    ax.scatter(forest_days["date"], forest_days[column], s=12, marker="x", label="isolation forest", zorder=3)
    ax.set_ylabel(f"{column} (GW, daily mean)")
axes[0].legend()
plt.show()

### Flagged days in context (two weeks either side, flagged day shaded)

In [ ]:
def plot_flag_context(hourly, flagged_date, title):
    """Plot hourly load, wind and solar around `flagged_date`, shading the flagged day."""
    window = hourly.filter(
        pl.col("date").is_between(
            flagged_date - timedelta(days=NUM_CONTEXT_DAYS),
            flagged_date + timedelta(days=NUM_CONTEXT_DAYS),
        )
    )
    wind = (window["wind_onshore"] + window["wind_offshore"]) / 1000

    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(window["timestamp"], window["load_forecast"] / 1000, label="load", linewidth=0.8)
    ax.plot(window["timestamp"], wind, label="wind", linewidth=0.8)
    ax.plot(window["timestamp"], window["solar"] / 1000, label="solar", linewidth=0.8)
    day_start = window.filter(pl.col("date") == flagged_date)["timestamp"].min()
    ax.axvspan(day_start, day_start + timedelta(days=1), color="red", alpha=0.15)
    ax.set_ylabel("GW")
    ax.set_title(title)
    ax.legend(loc="upper right")
    plt.show()


strongest_z = flags.filter(pl.col("z_flag")).sort("z_score", descending=True).head(NUM_EXAMPLE_FLAGS)
for row in strongest_z.iter_rows(named=True):
    plot_flag_context(hourly, row["date"], f"z-score flag {row['date']} (score {row['z_score']:.1f})")

In [ ]:
# Lower isolation-forest scores mean more anomalous.
strongest_forest = flags.filter(pl.col("forest_flag")).sort("forest_score").head(NUM_EXAMPLE_FLAGS)
for row in strongest_forest.iter_rows(named=True):
    plot_flag_context(hourly, row["date"], f"isolation forest flag {row['date']} (score {row['forest_score']:.3f})")